# LaTeX Table Exporter

## Purpose
Convert evaluation results and leaderboards to publication-ready LaTeX tables.
Supports booktabs formatting and automatic rounding.

## Inputs
- CSV files from `/work/eval/` (leaderboards, metrics summaries)
- DataFrames from other analysis notebooks

## Outputs
- `.tex` files with formatted LaTeX tables
- Console preview of LaTeX code

## What This Notebook Does
1. Defines helper functions for LaTeX conversion:
   - `df_to_latex_string()`: Convert DataFrame to LaTeX with options
   - `_normalize_df_for_export()`: Handle index formatting
2. Loads evaluation results from CSV files
3. Formats numeric columns with specified precision
4. Generates LaTeX tables with:
   - Booktabs formatting (toprule, midrule, bottomrule)
   - Optional captions and labels
   - Escaped special characters
5. Saves tables for paper inclusion

In [1]:
from pathlib import Path
import pandas as pd

def _normalize_df_for_export(df: pd.DataFrame, index_policy: str = "auto") -> pd.DataFrame:
    """
    Ensure the index isn't accidentally dropped:
    - index_policy='auto': if the index has a name or looks nontrivial, reset it to a column.
    - index_policy='keep': keep the index as-is.
    - index_policy='reset': always reset index to a column.
    """
    if index_policy not in {"auto", "keep", "reset"}:
        raise ValueError("index_policy must be one of {'auto','keep','reset'}")

    if index_policy == "keep":
        return df

    if index_policy == "reset":
        return df.reset_index()

    # auto: reset if index is not a simple RangeIndex or has a name
    if not isinstance(df.index, pd.RangeIndex) or (df.index.name is not None):
        return df.reset_index()
    return df


def df_to_latex_string(
    df: pd.DataFrame,
    caption: str | None = None,
    label: str | None = None,
    round_digits: int | None = 3,
    index: bool = False,
    use_booktabs: bool = True,
) -> str:
    """
    Convert a DataFrame to LaTeX. Uses DataFrame.to_latex for broad compatibility.
    (You can swap in Styler.to_latex if you need bolding or CSS conversion.)
    """
    # optional rounding for numeric columns
    if round_digits is not None:
        df = df.copy()
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].astype(float).round(round_digits)

    # Pandas writes booktabs if available; hrules are managed by to_latex itself.
    # float_format only affects formatting when round_digits is None; we already rounded.
    tex = df.to_latex(
        index=index,
        caption=caption,
        label=label,
        escape=False,
    )

    # If you want explicit \toprule/\midrule/\bottomrule control via Styler, switch to a Styler here.
    return tex


def save_tables_to_latex(
    tables,
    outdir: str | Path = "tables_tex",
    captions: dict | None = None,
    labels: dict | None = None,
    index_policy: str = "auto",   # 'auto' | 'keep' | 'reset'
    round_digits: int | None = 3,
    default_prefix: str = "",
):
    """
    Save multiple DataFrames to individual .tex files.
    - tables: dict[str, DataFrame] OR list[DataFrame]. If list, names become table_1, table_2, ...
    - captions/labels: optional dicts keyed by table name.
    - index_policy: how to handle index ('auto' recommended).
    - round_digits: round numeric columns (None to skip).
    - default_prefix: prefix added to filenames (e.g., 'exp1_').
    Returns: dict[name] = Path to .tex file
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    # normalize input into (name, df) pairs
    if isinstance(tables, dict):
        items = list(tables.items())
    else:
        # assume iterable of dataframes
        items = [(f"table_{i+1}", df) for i, df in enumerate(tables)]

    written = {}
    for name, df in items:
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Entry '{name}' is not a pandas DataFrame")

        df_export = _normalize_df_for_export(df, index_policy=index_policy)
        # Keep index=False because we normalized already
        tex = df_to_latex_string(
            df_export,
            caption=(captions or {}).get(name),
            label=(labels or {}).get(name),
            round_digits=round_digits,
            index=False,
        )

        fname = f"{default_prefix}{name}.tex"
        path = outdir / fname
        path.write_text(tex, encoding="utf-8")
        written[name] = path

    return written


In [10]:
FILE_DIR = Path("/work/eval/bootstrap_lexical")
EVAL_OUT  = Path("/work/eval")
FILE_NAME = "bootstrap_sigtests_pairs.csv"

# Load your CSV into a DataFrame
df = pd.read_csv(FILE_DIR / FILE_NAME)

# Wrap it into a dict with a name (this becomes the filename)
tables = {FILE_NAME : df}

# (Optional) add caption and label for LaTeX
captions = {"my_table": "Results from uploaded CSV."}
labels = {"my_table": "tab:my_table"}

# Now save using the helper we wrote earlier
written = save_tables_to_latex(
    tables,
    outdir=EVAL_OUT,      # output folder
    captions=captions,
    labels=labels,
    index_policy="auto",      # reset index if necessary
    round_digits=3
)

for name, path in written.items():
    print(f"Saved {name} -> {path}")


Saved bootstrap_sigtests_pairs.csv -> /work/eval/bootstrap_sigtests_pairs.csv.tex
